# 07 - Prompt Engineering and Generation

## Definition
Prompt engineering defines behavior contracts for grounded, citation-aware generation.

## Theory
Instruction clarity and constraints reduce unsupported claims and improve consistency.

## Motivation
A high-recall retriever still fails if prompts allow context-ignoring behavior.

## Architecture
Retrieved context + policy instructions + question -> constrained generation.

## Real-world Examples
- Enterprise support assistant over product docs
- Compliance assistant over policy text
- Research assistant over paper abstracts

## Best Practices
- Measure retrieval and generation separately
- Track latency and grounding together
- Keep prompts strict about citations and uncertainty

## Common Mistakes
- Skipping retrieval diagnostics
- Using mismatched embedding models for index/query
- Reporting only one metric and claiming broad quality


In [ ]:
from pathlib import Path
import json
import pandas as pd

from rag_system import (
    RAGConfig,
    ProjectRunner,
    ChunkingStrategy,
)

cfg = RAGConfig(project_root=Path(".."), profile="balanced")
runner = ProjectRunner(cfg)


In [ ]:
from rag_system.prompts import PromptLibrary

query = 'Explain retrieval-augmented generation in simple terms.'
context = '[1] RAG retrieves relevant documents before generation.'

bad = PromptLibrary.bad_prompt_example(query, context)
good = PromptLibrary.good_prompt_example(query, context)
bad[:300], good[:300]


In [ ]:
bundle = runner.ingest_dataset()
chunking = runner.run_chunking(bundle['documents'][:650], strategy=ChunkingStrategy.RECURSIVE)
runner.index_chunks(chunking.chunks, reset=True)

result = runner.pipeline.answer(bundle['queries'][10].query, top_k=6)
{
    'answer': result.answer[:420],
    'citations': result.citations,
    'abstained': result.abstained,
    'abstain_reason': result.abstain_reason,
}


## Code Explanation
The prompt library enforces context-only answering and explicit abstention when evidence is insufficient.